# RelShift Faz 2 — Orbit Explainability Deney Notebooku

Bu notebook, Faz 2'nin eksik kalan **gerçek deneylerini** uçtan uca üretir:

1. Cora ve CiteSeer üzerinde uniform exact RelShift pruning matrisi  
2. GCN ve GraphSAGE eğitimleri  
3. Orbit-sensitivity veri seti  
4. Leakage-safe controlled regression  
5. Cross-dataset ve cross-model orbit ağırlıkları  
6. Uniform–weighted RelShift karşılaştırması  
7. Leave-one-orbit ve leave-group-out deneyleri  
8. Guard ablation  
9. Kabul kontrolleri, özet tablolar ve Drive'a export

Notebook, mevcut repo scriptlerini ve artifact sözleşmesini kullanır. Büyük deneyler tekrar başlatıldığında tamamlanmış training/pruning çıktıları atlanır.

> `pilot` profili yalnızca hattı doğrulamak içindir. Makale sonuçları için `paper` profili kullanılmalıdır.

## 0. Çalışma prensibi

- **Pruning seed** ile **training seed** ayrıdır.
- Orbit sensitivity yalnızca **uniform RelShift** artefaktlarından öğrenilir.
- Aynı sparsified graph, regression train ve test tarafına aynı anda giremez.
- Learned weight'ler yalnız izin verilen training subsetinden türetilir.
- Cross-dataset ve cross-model değerlendirmede held-out veri ağırlık üretimine girmez.
- Büyük raw sonuçlar `results/phase2_experiments/` altında tutulur.

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import shlex
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Iterable, Mapping, Sequence

import numpy as np
import pandas as pd
import yaml

try:
    from IPython.display import display
except Exception:
    display = print

IN_COLAB = "google.colab" in sys.modules
print({"python": sys.version, "in_colab": IN_COLAB})

## 1. Google Drive ve repository ayarları

In [ ]:
MOUNT_DRIVE = True
GITHUB_REPO = "https://github.com/sasipilav/BBM462-Structure-Faithful-GNN-Sparsification"
GITHUB_BRANCH = "main"
WORKDIR = Path("/content/structure_faithful_gnn" if IN_COLAB else "./structure_faithful_gnn").resolve()

# Colab secret adı. Public repo için token zorunlu değildir.
GITHUB_TOKEN_SECRET_NAME = "GITHUB_TOKEN"

if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_EXPORT_DIR = Path(
    "/content/drive/MyDrive/RelShift_phase2_outputs"
    if IN_COLAB
    else "./RelShift_phase2_outputs"
).resolve()

print({
    "repo": GITHUB_REPO,
    "branch": GITHUB_BRANCH,
    "workdir": str(WORKDIR),
    "drive_export": str(DRIVE_EXPORT_DIR),
})

## 2. Deney profili ve ana parametreler

In [ ]:
# Önce "pilot" çalıştırın. Hattın tamamı bittikten sonra "paper" seçin.
RUN_PROFILE = "pilot"  # "pilot" veya "paper"

PROFILES = {
    "pilot": {
        "datasets": ["cora", "citeseer"],
        "models": ["gcn", "graphsage"],
        "rhos": [0.05, 0.20],
        "pruning_seeds": [0, 1],
        "training_seeds": [0, 1],
        "ablation_pruning_seeds": [0],
        "ablation_training_seeds": [0],
        "run_all_leave_orbit": False,
        "run_all_leave_group": True,
    },
    "paper": {
        "datasets": ["cora", "citeseer"],
        "models": ["gcn", "graphsage"],
        "rhos": [0.05, 0.10, 0.15, 0.20, 0.30],
        "pruning_seeds": [0, 1, 2, 3, 4],
        "training_seeds": [0, 1, 2, 3, 4],
        # Full ablation çok büyüktür; gerekirse 0,1,2'ye çıkarın.
        "ablation_pruning_seeds": [0, 1, 2],
        "ablation_training_seeds": [0, 1, 2],
        "run_all_leave_orbit": True,
        "run_all_leave_group": True,
    },
}

if RUN_PROFILE not in PROFILES:
    raise ValueError(f"RUN_PROFILE must be one of {sorted(PROFILES)}")

P = PROFILES[RUN_PROFILE]
DATASETS = list(P["datasets"])
MODELS = list(P["models"])
RHOS = list(P["rhos"])
PRUNING_SEEDS = list(P["pruning_seeds"])
TRAINING_SEEDS = list(P["training_seeds"])
ABLATION_PRUNING_SEEDS = list(P["ablation_pruning_seeds"])
ABLATION_TRAINING_SEEDS = list(P["ablation_training_seeds"])

# Stage anahtarları. Bir aşamayı tamamladıktan sonra False yaparak notebook'u
# güvenli biçimde devam ettirebilirsiniz.
RUN_SETUP = True
RUN_REPO_TESTS = True
RUN_UNIFORM_PRUNING = True
RUN_UNIFORM_TRAINING = True
RUN_SENSITIVITY_DATASET = True
RUN_CONTROLLED_REGRESSION = True
RUN_LEARNED_WEIGHTS = True
RUN_WEIGHTED_EVALUATION = True
RUN_LEAVE_OUT_ABLATIONS = True
RUN_GUARD_ABLATION = True
RUN_FINAL_AUDIT = True
RUN_EXPORT = True

# Existing outputs are reused. True yaparsanız tüm Faz-2 deney klasörü silinir.
RESET_PHASE2_RESULTS = False

# Target matching. Exact RelShift'te round(m*rho) sebebiyle yarım-edge farkı
# normaldir; daha büyük fark veya saturation final analizde reddedilir.
MAX_TARGET_GAP = 0.005

# Controlled regression
REGRESSION_TARGETS = ["accuracy_loss", "macro_f1_loss"]
ORBIT_FEATURE_FAMILY = "standardized_absolute"
REGRESSION_ESTIMATORS = ["ridge", "elastic_net"]
ARTIFACT_FOLDS = 3
PERMUTATION_REPEATS = 16 if RUN_PROFILE == "pilot" else 64
BOOTSTRAP_REPEATS = 100 if RUN_PROFILE == "pilot" else 500
RANDOM_SEED = 20260701

# Learned weight üretimi. Negatif katsayılar yapısal zarar cezası olarak
# kullanılmaz. Hiç pozitif katsayı kalmazsa aşama hata verir; sessiz fallback yoktur.
WEIGHT_TARGET = "accuracy_loss"
WEIGHT_TRANSFORM = "positive"
WEIGHT_NORMALIZATION = "mean_one"
MINIMUM_SIGN_CONSISTENCY = 0.60
WEIGHT_FLOOR = 0.0

DATASET_CONFIGS = {
    "cora": "configs/datasets/cora.yaml",
    "citeseer": "configs/datasets/citeseer.yaml",
}
MODEL_CONFIGS = {
    "gcn": "configs/models/gcn.yaml",
    "graphsage": "configs/models/graphsage.yaml",
}

BASE_EXPLAINABILITY_CONFIG = "configs/pruning/relshift_incremental_exact_orbit_explainability.yaml"
WEIGHTED_TEMPLATE_CONFIG = "configs/pruning/relshift_incremental_exact_orbit_weighted_template.yaml"

RESULTS_ROOT = WORKDIR / "results" / "phase2_experiments"
UNIFORM_FRONTIER_ROOT = RESULTS_ROOT / "uniform_frontiers"
UNIFORM_MANIFEST_DIR = RESULTS_ROOT / "uniform_manifest"
UNIFORM_TRAINING_ROOT = RESULTS_ROOT / "uniform_training"
DENSE_ROOT = RESULTS_ROOT / "dense"
SENSITIVITY_ROOT = RESULTS_ROOT / "sensitivity"
REGRESSION_ROOT = RESULTS_ROOT / "regression"
WEIGHTS_ROOT = RESULTS_ROOT / "weights"
WEIGHTED_ROOT = RESULTS_ROOT / "weighted_evaluation"
ABLATION_ROOT = RESULTS_ROOT / "leave_out_ablations"
GUARD_ROOT = RESULTS_ROOT / "guard_ablation"
GENERATED_CONFIG_ROOT = RESULTS_ROOT / "generated_configs"
REPORT_ROOT = RESULTS_ROOT / "reports"

print({
    "profile": RUN_PROFILE,
    "datasets": DATASETS,
    "models": MODELS,
    "rhos": RHOS,
    "pruning_seeds": PRUNING_SEEDS,
    "training_seeds": TRAINING_SEEDS,
})

### 2.1. Koşu sayısını başlamadan hesapla

In [ ]:
def matrix_counts(
    variants: int,
    datasets: int,
    rhos: int,
    pruning_seeds: int,
    models: int,
    training_seeds: int,
) -> dict[str, int]:
    pruning = variants * datasets * rhos * pruning_seeds
    training = pruning * models * training_seeds
    return {"pruning_artifacts": pruning, "gnn_training_runs": training}

uniform_counts = matrix_counts(
    1, len(DATASETS), len(RHOS), len(PRUNING_SEEDS), len(MODELS), len(TRAINING_SEEDS)
)

leave_variant_count = (
    1
    + (15 if P["run_all_leave_orbit"] else 0)
    + (6 if P["run_all_leave_group"] else 0)
)
ablation_counts = matrix_counts(
    leave_variant_count,
    len(DATASETS),
    len(RHOS),
    len(ABLATION_PRUNING_SEEDS),
    len(MODELS),
    len(ABLATION_TRAINING_SEEDS),
)
guard_counts = matrix_counts(
    4,
    len(DATASETS),
    len(RHOS),
    len(ABLATION_PRUNING_SEEDS),
    len(MODELS),
    len(ABLATION_TRAINING_SEEDS),
)

display(pd.DataFrame([
    {"stage": "uniform sensitivity matrix", **uniform_counts},
    {"stage": "leave-out suite", **ablation_counts},
    {"stage": "guard ablation", **guard_counts},
]))

if RUN_PROFILE == "pilot":
    print("UYARI: pilot sonuçlarını makale sonucu olarak kullanmayın.")

## 3. Yardımcı fonksiyonlar

In [ ]:
def q(value: object) -> str:
    return shlex.quote(str(value))

def run(
    command: Sequence[object] | str,
    *,
    cwd: Path | str | None = None,
    env: Mapping[str, str] | None = None,
    check: bool = True,
    capture: bool = False,
) -> subprocess.CompletedProcess:
    if isinstance(command, str):
        printable = command
        shell = True
        args = command
    else:
        args = [str(item) for item in command]
        printable = " ".join(q(item) for item in args)
        shell = False
    print(f"\n+ {printable}")
    completed = subprocess.run(
        args,
        cwd=None if cwd is None else str(cwd),
        env=None if env is None else dict(env),
        text=True,
        capture_output=capture,
        shell=shell,
    )
    if capture and completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        if check:
            raise subprocess.CalledProcessError(
                completed.returncode, completed.args, completed.stdout, completed.stderr
            )
    elif capture and completed.stderr:
        print(completed.stderr)
    return completed

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def write_json(path: Path, payload: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str) + "\n", encoding="utf-8")

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def resolve_optional_github_token() -> str | None:
    if not IN_COLAB:
        return None
    try:
        from google.colab import userdata
        return userdata.get(GITHUB_TOKEN_SECRET_NAME)
    except Exception:
        return None

def prepare_repo() -> None:
    token = resolve_optional_github_token()
    env = os.environ.copy()
    askpass = None
    if token:
        askpass = Path("/tmp/relshift_git_askpass.sh")
        askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n', encoding="utf-8")
        askpass.chmod(0o700)
        env["GIT_ASKPASS"] = str(askpass)
        env["GITHUB_TOKEN"] = token

    if WORKDIR.exists() and (WORKDIR / ".git").exists():
        run(["git", "-C", WORKDIR, "fetch", "origin", GITHUB_BRANCH], env=env)
        run(["git", "-C", WORKDIR, "checkout", GITHUB_BRANCH], env=env)
        run(["git", "-C", WORKDIR, "pull", "--ff-only", "origin", GITHUB_BRANCH], env=env)
    elif WORKDIR.exists():
        raise RuntimeError(
            f"{WORKDIR} exists but is not a Git checkout. Remove it or change WORKDIR."
        )
    else:
        run([
            "git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH,
            GITHUB_REPO, WORKDIR
        ], env=env)

def add_repo_to_path() -> None:
    src = str(WORKDIR / "src")
    root = str(WORKDIR)
    for item in (src, root):
        if item not in sys.path:
            sys.path.insert(0, item)

def checkpoint_results(tag: str) -> Path:
    DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    archive_base = DRIVE_EXPORT_DIR / f"phase2_{RUN_PROFILE}_{tag}"
    archive = Path(shutil.make_archive(
        str(archive_base), "zip",
        root_dir=RESULTS_ROOT.parent,
        base_dir=RESULTS_ROOT.name,
    ))
    print({"checkpoint": str(archive), "sha256": sha256_file(archive)})
    return archive

## 4. Repository kurulumu, bağımlılıklar, ORCA ve native extension

In [ ]:
if RUN_SETUP:
    prepare_repo()
    add_repo_to_path()

    run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKDIR)

    # ORCA
    orca_dir = WORKDIR / "tmp" / "orca"
    orca_cpp = orca_dir / "orca.cpp"
    orca_bin = orca_dir / "orca"
    if not orca_cpp.exists():
        orca_dir.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--depth", "1", "https://github.com/thocevar/orca", orca_dir])
    if not orca_bin.exists() or orca_bin.stat().st_mtime < orca_cpp.stat().st_mtime:
        run(["g++", "-O3", "-std=c++11", "-o", orca_bin, orca_cpp], cwd=orca_dir)
    os.environ["ORCA_BINARY"] = str(orca_bin)

    run(
        [sys.executable, "scripts/build_relshift_incremental_extension.py", "--smoke"],
        cwd=WORKDIR,
    )

    print({
        "repo_commit": run(
            ["git", "-C", WORKDIR, "rev-parse", "HEAD"],
            capture=True,
        ).stdout.strip(),
        "orca": str(orca_bin),
    })
else:
    add_repo_to_path()

if RESET_PHASE2_RESULTS and RESULTS_ROOT.exists():
    shutil.rmtree(RESULTS_ROOT)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

## 5. Repository doğruluk testleri

In [ ]:
if RUN_REPO_TESTS:
    # Önce Faz 2 hedefli testler, sonra tüm suite.
    run([
        sys.executable, "-m", "pytest", "-q",
        "tests/test_orbit_registry.py",
        "tests/test_orbit_explainability.py",
        "tests/test_orbit_checkpoints.py",
        "tests/test_orbit_sensitivity.py",
        "tests/test_orbit_weighted_relshift.py",
    ], cwd=WORKDIR)
    run([sys.executable, "-m", "pytest", "-q"], cwd=WORKDIR)

## 6. Pruning ve training yardımcıları

Bu hücre:

- Her pruning seed için exact RelShift calibration çalıştırır.
- Hedef reduction noktalarını tek manifestte birleştirir.
- Saturated veya hedefe uzak artefaktları reddeder.
- Aynı pruning artefaktının iki target için seçilmesini reddeder.
- Dense ve pruned training koşularını resume-safe biçimde yürütür.

In [ ]:
def dataset_flag_values(datasets: Sequence[str], rhos: Sequence[float]) -> list[str]:
    payload = ",".join(f"{rho:g}" for rho in rhos)
    flags: list[str] = []
    for dataset in datasets:
        flags.extend(["--comparison-target-set", f"{dataset}:{payload}"])
        flags.extend(["--rho-grid-set", f"{dataset}:{payload}"])
    return flags

def build_phase2_manifest(
    matched_paths: Sequence[Path],
    *,
    variant: str,
    output_path: Path,
    max_target_gap: float = MAX_TARGET_GAP,
) -> Path:
    records: list[dict[str, object]] = []
    seen_artifacts: set[str] = set()

    for matched_path in matched_paths:
        frame = pd.read_csv(matched_path)
        required = {
            "dataset", "method", "target_edge_reduction", "achieved_edge_reduction",
            "abs_gap", "pruning_seed", "target_rho", "saturated", "run_dir"
        }
        missing = required.difference(frame.columns)
        if missing:
            raise ValueError(f"{matched_path} missing columns: {sorted(missing)}")

        for row in frame.to_dict(orient="records"):
            dataset = str(row["dataset"]).lower()
            if dataset not in DATASET_CONFIGS:
                raise ValueError(f"Unknown dataset {dataset}")
            gap = float(row["abs_gap"])
            if gap > max_target_gap:
                raise ValueError(
                    f"{variant}: target gap {gap:.6f} exceeds {max_target_gap} "
                    f"for {dataset}, target={row['target_edge_reduction']}"
                )
            if bool(row["saturated"]):
                raise ValueError(
                    f"{variant}: saturated artifact cannot enter final matrix: {row['run_dir']}"
                )
            artifact = str(Path(row["run_dir"]).resolve())
            if artifact in seen_artifacts:
                raise ValueError(
                    f"{variant}: one pruning artifact was matched to multiple targets: {artifact}"
                )
            seen_artifacts.add(artifact)

            pruning_seed = int(row["pruning_seed"])
            target = float(row["target_edge_reduction"])
            run_name = (
                f"{variant}__pseed_{pruning_seed}"
                f"__target_{target:.6f}".replace(".", "p")
            )
            records.append({
                "dataset": dataset,
                "dataset_config": str((WORKDIR / DATASET_CONFIGS[dataset]).resolve()),
                "status": "phase2_matrix",
                "method": "relshift",
                "target_edge_reduction": target,
                "achieved_edge_reduction": float(row["achieved_edge_reduction"]),
                "artifact_run_dir": artifact,
                "target_rho": float(row["target_rho"]),
                "pruning_seed": pruning_seed,
                "variant": variant,
                "run_name": run_name,
            })

    result = pd.DataFrame(records).sort_values(
        ["dataset", "target_edge_reduction", "pruning_seed"]
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False)
    write_json(output_path.with_suffix(".json"), {"rows": result.to_dict(orient="records")})
    return output_path

def run_relshift_matrix(
    *,
    config_path: Path | str,
    variant: str,
    datasets: Sequence[str],
    rhos: Sequence[float],
    pruning_seeds: Sequence[int],
    frontier_root: Path,
    manifest_path: Path,
) -> Path:
    matched_paths: list[Path] = []
    config_path = Path(config_path)
    if not config_path.is_absolute():
        config_path = WORKDIR / config_path

    for pruning_seed in pruning_seeds:
        seed_root = frontier_root / f"pruning_seed_{pruning_seed}"
        cmd = [
            sys.executable,
            "scripts/run_relshift_calibration.py",
            "--datasets",
            *[DATASET_CONFIGS[name] for name in datasets],
            "--pruning",
            str(config_path),
            "--seed",
            str(pruning_seed),
            *dataset_flag_values(datasets, rhos),
            "--output-root",
            str(seed_root),
        ]
        run(cmd, cwd=WORKDIR)
        matched = seed_root / "matched_targets.csv"
        if not matched.exists():
            raise FileNotFoundError(matched)
        matched_paths.append(matched)

    return build_phase2_manifest(
        matched_paths,
        variant=variant,
        output_path=manifest_path,
    )

def expected_dense_dir(
    dense_root: Path, dataset: str, model: str, training_seed: int
) -> Path:
    return dense_root / dataset / model / "dense" / f"seed_{training_seed}"

def expected_pruned_training_dir(
    training_root: Path,
    dataset: str,
    model: str,
    training_seed: int,
    target_rho: float,
    run_name: str,
) -> Path:
    return (
        training_root / dataset / model / "relshift"
        / f"seed_{training_seed}-rho_{target_rho:g}" / run_name
    )

def train_manifest_resumable(
    *,
    manifest_path: Path,
    model_names: Sequence[str],
    training_seeds: Sequence[int],
    output_root: Path,
    dense_root: Path,
) -> Path:
    add_repo_to_path()
    from structure_faithful_gnn.config import load_dataset_config, load_model_config
    from structure_faithful_gnn.experiments.runner import (
        run_dense_experiment,
        run_training_on_pruned_artifact,
    )

    manifest = pd.read_csv(manifest_path)
    if manifest.empty:
        raise ValueError(f"Empty manifest: {manifest_path}")

    output_root.mkdir(parents=True, exist_ok=True)
    dense_root.mkdir(parents=True, exist_ok=True)
    summary_rows: list[dict[str, object]] = []

    for training_seed in training_seeds:
        for row in manifest.to_dict(orient="records"):
            dataset = str(row["dataset"])
            dataset_config = load_dataset_config(str(row["dataset_config"]))
            for model_name in model_names:
                model_config = load_model_config(
                    str((WORKDIR / MODEL_CONFIGS[model_name]).resolve())
                )

                dense_dir = expected_dense_dir(
                    dense_root, dataset, model_config.name, int(training_seed)
                )
                if not (dense_dir / "metrics.json").exists():
                    run_dense_experiment(
                        dataset_config,
                        model_config,
                        seed=int(training_seed),
                        output_root=dense_root,
                    )

                target_rho = float(row["target_rho"])
                run_name = str(row["run_name"])
                expected = expected_pruned_training_dir(
                    output_root,
                    dataset,
                    model_config.name,
                    int(training_seed),
                    target_rho,
                    run_name,
                )
                if (expected / "metrics.json").exists():
                    status = "reused"
                    training_run_dir = expected
                else:
                    outcome = run_training_on_pruned_artifact(
                        dataset_config,
                        model_config,
                        artifact_dir=str(row["artifact_run_dir"]),
                        seed=int(training_seed),
                        output_root=output_root,
                        run_name=run_name,
                    )
                    status = "trained"
                    training_run_dir = outcome.run_dir

                summary_rows.append({
                    **row,
                    "model": model_config.name,
                    "training_seed": int(training_seed),
                    "training_run_dir": str(Path(training_run_dir).resolve()),
                    "training_status": status,
                })
                gc.collect()
                try:
                    import torch
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                except Exception:
                    pass

    summary = pd.DataFrame(summary_rows)
    summary_path = output_root / "phase2_training_manifest_summary.csv"
    summary.to_csv(summary_path, index=False)
    write_json(
        output_root / "phase2_training_manifest_summary.json",
        {"rows": summary.to_dict(orient="records")},
    )
    return summary_path

def combine_manifests(paths: Sequence[Path], output_path: Path) -> Path:
    frames = [pd.read_csv(path) for path in paths]
    combined = pd.concat(frames, ignore_index=True)
    duplicate_cols = ["dataset", "variant", "pruning_seed", "target_rho", "artifact_run_dir"]
    duplicates = combined.duplicated(duplicate_cols, keep=False)
    if duplicates.any():
        display(combined.loc[duplicates, duplicate_cols])
        raise ValueError("Duplicate rows in combined manifest.")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    combined.to_csv(output_path, index=False)
    return output_path

## 7. Uniform exact RelShift pruning matrisi

In [ ]:
uniform_manifest = UNIFORM_MANIFEST_DIR / "uniform_phase2_manifest.csv"

if RUN_UNIFORM_PRUNING:
    uniform_manifest = run_relshift_matrix(
        config_path=BASE_EXPLAINABILITY_CONFIG,
        variant="uniform",
        datasets=DATASETS,
        rhos=RHOS,
        pruning_seeds=PRUNING_SEEDS,
        frontier_root=UNIFORM_FRONTIER_ROOT,
        manifest_path=uniform_manifest,
    )

uniform_frame = pd.read_csv(uniform_manifest)
display(uniform_frame.head())
print({
    "uniform_artifacts": len(uniform_frame),
    "datasets": sorted(uniform_frame["dataset"].unique()),
    "pruning_seeds": sorted(uniform_frame["pruning_seed"].unique()),
})

## 8. Uniform graphlar üzerinde GCN/GraphSAGE eğitim matrisi

In [ ]:
if RUN_UNIFORM_TRAINING:
    uniform_training_summary = train_manifest_resumable(
        manifest_path=uniform_manifest,
        model_names=MODELS,
        training_seeds=TRAINING_SEEDS,
        output_root=UNIFORM_TRAINING_ROOT,
        dense_root=DENSE_ROOT,
    )
    display(pd.read_csv(uniform_training_summary).head())

checkpoint_results("uniform_matrix")

## 9. Orbit sensitivity veri setini oluştur

In [ ]:
SENSITIVITY_DATASET_DIR = SENSITIVITY_ROOT / "dataset"
SENSITIVITY_CSV = SENSITIVITY_DATASET_DIR / "orbit_sensitivity_dataset.csv"

if RUN_SENSITIVITY_DATASET:
    run([
        sys.executable,
        "scripts/build_orbit_sensitivity_dataset.py",
        "--training-root", str(UNIFORM_TRAINING_ROOT),
        "--dense-root", str(DENSE_ROOT),
        "--output-dir", str(SENSITIVITY_DATASET_DIR),
    ], cwd=WORKDIR)

sensitivity = pd.read_csv(SENSITIVITY_CSV)
display(sensitivity[
    [
        "dataset", "model", "training_seed", "pruning_seed",
        "target_rho", "achieved_edge_reduction",
        "accuracy_loss", "macro_f1_loss"
    ]
].head())

print({
    "rows": len(sensitivity),
    "artifacts": sensitivity["pruning_artifact_id"].nunique(),
    "datasets": sensitivity["dataset"].nunique(),
    "models": sensitivity["model"].nunique(),
    "training_seeds": sensitivity["training_seed"].nunique(),
    "pruning_seeds": sensitivity["pruning_seed"].nunique(),
})

## 10. Leakage-safe controlled regression

In [ ]:
def valid_split_modes(frame: pd.DataFrame) -> list[str]:
    candidates = {
        "dataset": "dataset",
        "model": "model",
        "reduction_band": "reduction_band",
        "pruning_seed": "pruning_seed",
    }
    modes = [
        mode for mode, column in candidates.items()
        if column in frame and frame[column].astype(str).nunique() >= 2
    ]
    modes.append("pruning_artifact")
    return modes

regression_outputs: dict[str, Path] = {}

if RUN_CONTROLLED_REGRESSION:
    split_modes = valid_split_modes(sensitivity)
    for target in REGRESSION_TARGETS:
        output_dir = REGRESSION_ROOT / target
        run([
            sys.executable,
            "scripts/run_orbit_sensitivity_regression.py",
            "--dataset", str(SENSITIVITY_CSV),
            "--output-dir", str(output_dir),
            "--target", target,
            "--split-modes", *split_modes,
            "--orbit-feature-family", ORBIT_FEATURE_FAMILY,
            "--estimators", *REGRESSION_ESTIMATORS,
            "--artifact-folds", str(ARTIFACT_FOLDS),
            "--permutation-repeats", str(PERMUTATION_REPEATS),
            "--bootstrap-repeats", str(BOOTSTRAP_REPEATS),
            "--random-seed", str(RANDOM_SEED),
        ], cwd=WORKDIR)
        regression_outputs[target] = output_dir

for target in REGRESSION_TARGETS:
    fold_path = REGRESSION_ROOT / target / "fold_metrics.csv"
    if fold_path.exists():
        fold = pd.read_csv(fold_path)
        summary = (
            fold.groupby(["split_mode", "estimator", "feature_set"], dropna=False)
            .agg(mae=("mae", "mean"), rmse=("rmse", "mean"), folds=("split_id", "nunique"))
            .reset_index()
        )
        print(f"\n### {target}")
        display(summary)

### 10.1. Orbitlerin generic graph kontrollerine ek açıklama gücü

In [ ]:
def incremental_value_table(fold_metrics_path: Path) -> pd.DataFrame:
    frame = pd.read_csv(fold_metrics_path)
    grouped = (
        frame.groupby(["split_mode", "estimator", "feature_set"], as_index=False)["mae"]
        .mean()
    )
    pivot = grouped.pivot_table(
        index=["split_mode", "estimator"],
        columns="feature_set",
        values="mae",
    ).reset_index()
    if {"controls_only", "orbits_plus_controls"}.issubset(pivot.columns):
        pivot["mae_improvement_from_orbits"] = (
            pivot["controls_only"] - pivot["orbits_plus_controls"]
        )
    return pivot

for target in REGRESSION_TARGETS:
    path = REGRESSION_ROOT / target / "fold_metrics.csv"
    if path.exists():
        print(target)
        display(incremental_value_table(path))

## 11. Leakage-free learned orbit weight üretimi

In [ ]:
def train_weight_from_subset(
    *,
    source_frame: pd.DataFrame,
    name: str,
    subset_description: dict[str, object],
) -> Path:
    add_repo_to_path()
    from structure_faithful_gnn.analysis.orbit_sensitivity import (
        run_controlled_orbit_regression,
    )
    from structure_faithful_gnn.pruning.orbit_weights import (
        derive_orbit_weight_spec_from_table,
        write_orbit_weight_spec,
    )

    if source_frame["pruning_artifact_id"].nunique() < 4:
        raise ValueError(
            f"{name}: at least four pruning artifacts are required for a useful learned weight."
        )

    subset_dir = WEIGHTS_ROOT / name
    subset_dir.mkdir(parents=True, exist_ok=True)
    subset_csv = subset_dir / "training_subset.csv"
    source_frame.to_csv(subset_csv, index=False)

    modes = valid_split_modes(source_frame)
    # Held-out dimension is not present in the subset; remaining dimensions are used.
    regression_result = run_controlled_orbit_regression(
        dataset_path=subset_csv,
        output_dir=subset_dir / "regression",
        target=WEIGHT_TARGET,
        split_modes=modes,
        orbit_feature_family=ORBIT_FEATURE_FAMILY,
        estimators=("ridge",),
        artifact_folds=ARTIFACT_FOLDS,
        permutation_repeats=PERMUTATION_REPEATS,
        bootstrap_repeats=BOOTSTRAP_REPEATS,
        random_seed=RANDOM_SEED,
    )

    spec = derive_orbit_weight_spec_from_table(
        table_path=regression_result.coefficient_stability_path,
        value_column="coefficient_median",
        filters={
            "target": WEIGHT_TARGET,
            "orbit_feature_family": ORBIT_FEATURE_FAMILY,
            "feature_set": "orbits_plus_controls",
            "estimator": "ridge",
        },
        transform=WEIGHT_TRANSFORM,
        aggregation="median",
        normalization=WEIGHT_NORMALIZATION,
        minimum_sign_consistency=MINIMUM_SIGN_CONSISTENCY,
        floor=WEIGHT_FLOOR,
    )
    spec_path = write_orbit_weight_spec(subset_dir / "orbit_weights.json", spec)
    write_json(
        subset_dir / "training_provenance.json",
        {
            "name": name,
            "subset_description": subset_description,
            "rows": len(source_frame),
            "pruning_artifacts": source_frame["pruning_artifact_id"].nunique(),
            "datasets": sorted(source_frame["dataset"].unique()),
            "models": sorted(source_frame["model"].unique()),
            "weight_spec": str(spec_path),
        },
    )
    return Path(spec_path)

weight_specs: dict[str, Path] = {}

if RUN_LEARNED_WEIGHTS:
    # Cross-dataset: A'da öğren, B'de değerlendir.
    if set(DATASETS) >= {"cora", "citeseer"}:
        for train_dataset, test_dataset in (("cora", "citeseer"), ("citeseer", "cora")):
            subset = sensitivity[sensitivity["dataset"] == train_dataset].copy()
            name = f"dataset_{train_dataset}_to_{test_dataset}"
            weight_specs[name] = train_weight_from_subset(
                source_frame=subset,
                name=name,
                subset_description={
                    "train_dataset": train_dataset,
                    "held_out_dataset": test_dataset,
                },
            )

    # Cross-model: GCN'de öğren, GraphSAGE'de değerlendir ve tersi.
    if set(MODELS) >= {"gcn", "graphsage"}:
        for train_model, test_model in (("gcn", "graphsage"), ("graphsage", "gcn")):
            subset = sensitivity[sensitivity["model"] == train_model].copy()
            name = f"model_{train_model}_to_{test_model}"
            weight_specs[name] = train_weight_from_subset(
                source_frame=subset,
                name=name,
                subset_description={
                    "train_model": train_model,
                    "held_out_model": test_model,
                },
            )

print({name: str(path) for name, path in weight_specs.items()})

### 11.1. Öğrenilmiş ağırlıkları görüntüle

In [ ]:
weight_rows = []
for name, path in weight_specs.items():
    payload = read_json(path)
    for orbit_id, weight in enumerate(payload["weights"]):
        weight_rows.append({"weight_spec": name, "orbit_id": orbit_id, "weight": weight})

if weight_rows:
    weight_frame = pd.DataFrame(weight_rows)
    display(weight_frame.pivot(index="orbit_id", columns="weight_spec", values="weight"))

## 12. Learned-weighted RelShift'i held-out veri üzerinde değerlendir

In [ ]:
def write_weighted_config(
    *,
    weight_file: Path,
    name: str,
) -> Path:
    source = WORKDIR / WEIGHTED_TEMPLATE_CONFIG
    payload = yaml.safe_load(source.read_text(encoding="utf-8"))
    payload.pop("orbit_weights", None)
    payload["orbit_weight_file"] = str(weight_file.resolve())
    payload["orbit_weight_normalization"] = "none"
    output = GENERATED_CONFIG_ROOT / "weighted" / f"{name}.yaml"
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(yaml.safe_dump(payload, sort_keys=False), encoding="utf-8")
    return output

weighted_manifests: list[Path] = []

if RUN_WEIGHTED_EVALUATION:
    for name, weight_file in weight_specs.items():
        config = write_weighted_config(weight_file=weight_file, name=name)

        if name.startswith("dataset_"):
            train_dataset, test_dataset = name.removeprefix("dataset_").split("_to_")
            eval_datasets = [test_dataset]
            eval_models = MODELS
        elif name.startswith("model_"):
            train_model, test_model = name.removeprefix("model_").split("_to_")
            eval_datasets = DATASETS
            eval_models = [test_model]
        else:
            raise ValueError(name)

        manifest = WEIGHTED_ROOT / name / "manifest.csv"
        run_relshift_matrix(
            config_path=config,
            variant=name,
            datasets=eval_datasets,
            rhos=RHOS,
            pruning_seeds=PRUNING_SEEDS,
            frontier_root=WEIGHTED_ROOT / name / "frontiers",
            manifest_path=manifest,
        )
        train_manifest_resumable(
            manifest_path=manifest,
            model_names=eval_models,
            training_seeds=TRAINING_SEEDS,
            output_root=WEIGHTED_ROOT / name / "training",
            dense_root=DENSE_ROOT,
        )
        weighted_manifests.append(manifest)

checkpoint_results("weighted_evaluation")

### 12.1. Uniform–weighted karşılaştırma tablosu

In [ ]:
def collect_relshift_training_rows(root: Path) -> pd.DataFrame:
    add_repo_to_path()
    from structure_faithful_gnn.analysis.artifacts import (
        canonical_run_row,
        discover_method_run_dirs,
    )

    rows = []
    for run_dir in discover_method_run_dirs(root, method="relshift"):
        row = canonical_run_row(run_dir)
        run_name = Path(run_dir).name
        variant = run_name.split("__pseed_", 1)[0]
        row["variant"] = variant
        rows.append(row)
    return pd.DataFrame(rows)

uniform_metrics = collect_relshift_training_rows(UNIFORM_TRAINING_ROOT)

weighted_metric_frames = []
for name in weight_specs:
    root = WEIGHTED_ROOT / name / "training"
    if root.exists():
        weighted_metric_frames.append(collect_relshift_training_rows(root))

weighted_metrics = (
    pd.concat(weighted_metric_frames, ignore_index=True)
    if weighted_metric_frames
    else pd.DataFrame()
)

if not weighted_metrics.empty:
    key = ["dataset", "model", "seed", "pruning_seed", "target_rho"]
    baseline = uniform_metrics.rename(columns={
        "accuracy": "uniform_accuracy",
        "macro_f1": "uniform_macro_f1",
        "achieved_edge_reduction": "uniform_achieved_edge_reduction",
    })
    comparison = weighted_metrics.merge(
        baseline[key + [
            "uniform_accuracy", "uniform_macro_f1",
            "uniform_achieved_edge_reduction"
        ]],
        on=key,
        how="left",
        validate="many_to_one",
    )
    comparison["accuracy_gain_vs_uniform"] = (
        comparison["accuracy"] - comparison["uniform_accuracy"]
    )
    comparison["macro_f1_gain_vs_uniform"] = (
        comparison["macro_f1"] - comparison["uniform_macro_f1"]
    )
    comparison_path = REPORT_ROOT / "weighted_vs_uniform.csv"
    comparison_path.parent.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(comparison_path, index=False)

    summary = (
        comparison.groupby(["variant", "dataset", "model"], as_index=False)
        .agg(
            accuracy_gain_mean=("accuracy_gain_vs_uniform", "mean"),
            accuracy_gain_std=("accuracy_gain_vs_uniform", "std"),
            macro_f1_gain_mean=("macro_f1_gain_vs_uniform", "mean"),
            macro_f1_gain_std=("macro_f1_gain_vs_uniform", "std"),
            observations=("accuracy_gain_vs_uniform", "size"),
        )
    )
    display(summary)

## 13. Leave-one-orbit ve leave-group-out configlerini üret

In [ ]:
ABLATION_CONFIG_DIR = GENERATED_CONFIG_ROOT / "leave_out"

if RUN_LEAVE_OUT_ABLATIONS:
    cmd = [
        sys.executable,
        "scripts/build_orbit_ablation_configs.py",
        "--base-config", WEIGHTED_TEMPLATE_CONFIG,
        "--output-dir", str(ABLATION_CONFIG_DIR),
    ]
    if not P["run_all_leave_orbit"]:
        cmd.append("--skip-orbits")
    if not P["run_all_leave_group"]:
        cmd.append("--skip-groups")
    run(cmd, cwd=WORKDIR)

ablation_manifest_json = ABLATION_CONFIG_DIR / "orbit_ablation_manifest.json"
ablation_definition = read_json(ablation_manifest_json)
print({
    "variant_count": ablation_definition["variant_count"],
    "variants": [item["name"] for item in ablation_definition["variants"]],
})

## 14. Leave-out pruning ve training suite

In [ ]:
if RUN_LEAVE_OUT_ABLATIONS:
    manifest_paths: list[Path] = []
    for record in ablation_definition["variants"]:
        variant = str(record["name"])
        config_path = Path(record["path"])
        manifest_path = ABLATION_ROOT / variant / "manifest.csv"
        run_relshift_matrix(
            config_path=config_path,
            variant=variant,
            datasets=DATASETS,
            rhos=RHOS,
            pruning_seeds=ABLATION_PRUNING_SEEDS,
            frontier_root=ABLATION_ROOT / variant / "frontiers",
            manifest_path=manifest_path,
        )
        manifest_paths.append(manifest_path)

    combined_ablation_manifest = combine_manifests(
        manifest_paths,
        ABLATION_ROOT / "combined_manifest.csv",
    )
    train_manifest_resumable(
        manifest_path=combined_ablation_manifest,
        model_names=MODELS,
        training_seeds=ABLATION_TRAINING_SEEDS,
        output_root=ABLATION_ROOT / "training",
        dense_root=DENSE_ROOT,
    )

checkpoint_results("leave_out_ablations")

### 14.1. Leave-out sonuçlarını uniform baseline'a göre özetle

In [ ]:
if (ABLATION_ROOT / "training").exists():
    ablation_metrics = collect_relshift_training_rows(ABLATION_ROOT / "training")
    key = ["dataset", "model", "seed", "pruning_seed", "target_rho"]

    uniform_subset = uniform_metrics[
        uniform_metrics["seed"].isin(ABLATION_TRAINING_SEEDS)
        & uniform_metrics["pruning_seed"].isin(ABLATION_PRUNING_SEEDS)
    ].copy()

    uniform_subset = uniform_subset.rename(columns={
        "accuracy": "uniform_accuracy",
        "macro_f1": "uniform_macro_f1",
        "largest_component_ratio": "uniform_lcc",
    })

    ablation_comparison = ablation_metrics.merge(
        uniform_subset[key + [
            "uniform_accuracy", "uniform_macro_f1", "uniform_lcc"
        ]],
        on=key,
        how="left",
        validate="many_to_one",
    )
    ablation_comparison["accuracy_delta_vs_uniform"] = (
        ablation_comparison["accuracy"] - ablation_comparison["uniform_accuracy"]
    )
    ablation_comparison["macro_f1_delta_vs_uniform"] = (
        ablation_comparison["macro_f1"] - ablation_comparison["uniform_macro_f1"]
    )
    ablation_comparison["lcc_delta_vs_uniform"] = (
        ablation_comparison["largest_component_ratio"] - ablation_comparison["uniform_lcc"]
    )

    ablation_summary = (
        ablation_comparison.groupby(["variant", "dataset", "model"], as_index=False)
        .agg(
            accuracy_delta_mean=("accuracy_delta_vs_uniform", "mean"),
            macro_f1_delta_mean=("macro_f1_delta_vs_uniform", "mean"),
            lcc_delta_mean=("lcc_delta_vs_uniform", "mean"),
            observations=("accuracy_delta_vs_uniform", "size"),
        )
        .sort_values("accuracy_delta_mean")
    )
    REPORT_ROOT.mkdir(parents=True, exist_ok=True)
    ablation_comparison.to_csv(
        REPORT_ROOT / "leave_out_vs_uniform_rows.csv", index=False
    )
    ablation_summary.to_csv(
        REPORT_ROOT / "leave_out_vs_uniform_summary.csv", index=False
    )
    display(ablation_summary)

## 15. Guard ablation configleri

In [ ]:
def generate_guard_configs() -> dict[str, Path]:
    source = WORKDIR / BASE_EXPLAINABILITY_CONFIG
    base = yaml.safe_load(source.read_text(encoding="utf-8"))
    variants = {
        "guards_both": {"guard_bridges": True, "d_min": 2},
        "guard_bridge_only": {"guard_bridges": True, "d_min": 0},
        "guard_degree_only": {"guard_bridges": False, "d_min": 2},
        "guards_none": {"guard_bridges": False, "d_min": 0},
    }
    paths: dict[str, Path] = {}
    for name, changes in variants.items():
        payload = dict(base)
        payload.update(changes)
        path = GENERATED_CONFIG_ROOT / "guards" / f"{name}.yaml"
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(yaml.safe_dump(payload, sort_keys=False), encoding="utf-8")
        paths[name] = path
    return paths

guard_configs = generate_guard_configs()
print({name: str(path) for name, path in guard_configs.items()})

## 16. Guard pruning ve training suite

In [ ]:
if RUN_GUARD_ABLATION:
    guard_manifest_paths: list[Path] = []
    for variant, config_path in guard_configs.items():
        manifest_path = GUARD_ROOT / variant / "manifest.csv"
        run_relshift_matrix(
            config_path=config_path,
            variant=variant,
            datasets=DATASETS,
            rhos=RHOS,
            pruning_seeds=ABLATION_PRUNING_SEEDS,
            frontier_root=GUARD_ROOT / variant / "frontiers",
            manifest_path=manifest_path,
        )
        guard_manifest_paths.append(manifest_path)

    combined_guard_manifest = combine_manifests(
        guard_manifest_paths,
        GUARD_ROOT / "combined_manifest.csv",
    )
    train_manifest_resumable(
        manifest_path=combined_guard_manifest,
        model_names=MODELS,
        training_seeds=ABLATION_TRAINING_SEEDS,
        output_root=GUARD_ROOT / "training",
        dense_root=DENSE_ROOT,
    )

checkpoint_results("guard_ablation")

### 16.1. Guard sonuçlarını özetle

In [ ]:
if (GUARD_ROOT / "training").exists():
    guard_metrics = collect_relshift_training_rows(GUARD_ROOT / "training")
    guard_summary = (
        guard_metrics.groupby(["variant", "dataset", "model"], as_index=False)
        .agg(
            accuracy_mean=("accuracy", "mean"),
            accuracy_std=("accuracy", "std"),
            macro_f1_mean=("macro_f1", "mean"),
            macro_f1_std=("macro_f1", "std"),
            achieved_reduction_mean=("achieved_edge_reduction", "mean"),
            lcc_mean=("largest_component_ratio", "mean"),
            components_mean=("num_components", "mean"),
            isolated_ratio_delta_mean=("isolated_node_ratio_delta", "mean"),
            observations=("accuracy", "size"),
        )
    )
    REPORT_ROOT.mkdir(parents=True, exist_ok=True)
    guard_metrics.to_csv(REPORT_ROOT / "guard_ablation_rows.csv", index=False)
    guard_summary.to_csv(REPORT_ROOT / "guard_ablation_summary.csv", index=False)
    display(guard_summary)

## 17. Son bilimsel ve artifact kontrolleri

In [ ]:
def final_audit() -> dict[str, object]:
    report: dict[str, object] = {
        "profile": RUN_PROFILE,
        "repo_commit": run(
            ["git", "-C", WORKDIR, "rev-parse", "HEAD"],
            capture=True,
        ).stdout.strip(),
        "checks": {},
    }
    checks: dict[str, object] = report["checks"]  # type: ignore[assignment]

    # 1. Sensitivity satır sayısı ve artifact blocking
    frame = pd.read_csv(SENSITIVITY_CSV)
    checks["sensitivity_rows"] = len(frame)
    checks["sensitivity_artifacts"] = frame["pruning_artifact_id"].nunique()
    checks["duplicate_training_pairs"] = int(
        frame.duplicated(
            ["pruning_artifact_id", "dataset", "model", "training_seed"]
        ).sum()
    )
    if checks["duplicate_training_pairs"] != 0:
        raise AssertionError("Duplicate sensitivity training pairs detected.")

    # 2. Required dimensions
    for column in ("dataset", "model", "reduction_band", "pruning_seed"):
        checks[f"{column}_unique"] = int(frame[column].astype(str).nunique())

    # 3. Regression incremental value tables
    regression_checks = {}
    for target in REGRESSION_TARGETS:
        path = REGRESSION_ROOT / target / "fold_metrics.csv"
        if path.exists():
            table = incremental_value_table(path)
            regression_checks[target] = table.to_dict(orient="records")
    checks["regression_incremental_value"] = regression_checks

    # 4. Artifact manifests and checksums
    manifest_paths = sorted(RESULTS_ROOT.rglob("*manifest*.json"))
    checksum_errors = []
    for manifest_path in manifest_paths:
        try:
            json.loads(manifest_path.read_text(encoding="utf-8"))
        except Exception as exc:
            checksum_errors.append({"path": str(manifest_path), "error": str(exc)})
    checks["manifest_count"] = len(manifest_paths)
    checks["manifest_parse_errors"] = checksum_errors
    if checksum_errors:
        raise AssertionError(checksum_errors)

    # 5. Weight provenance
    checks["weight_specs"] = {
        name: {
            "path": str(path),
            "sha256": sha256_file(path),
            "payload": read_json(path),
        }
        for name, path in weight_specs.items()
        if path.exists()
    }

    REPORT_ROOT.mkdir(parents=True, exist_ok=True)
    write_json(REPORT_ROOT / "phase2_final_audit.json", report)
    return report

if RUN_FINAL_AUDIT:
    audit = final_audit()
    print(json.dumps(audit, indent=2, default=str)[:12000])

## 18. Makale için ana tabloları görüntüle

In [ ]:
report_candidates = [
    REPORT_ROOT / "weighted_vs_uniform.csv",
    REPORT_ROOT / "leave_out_vs_uniform_summary.csv",
    REPORT_ROOT / "guard_ablation_summary.csv",
]

for path in report_candidates:
    print(f"\n### {path.name}")
    if path.exists():
        display(pd.read_csv(path).head(50))
    else:
        print("Henüz üretilmedi.")

## 19. Sonuçları zipleyip Drive'a aktar

In [ ]:
if RUN_EXPORT:
    DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    archive_base = DRIVE_EXPORT_DIR / f"RelShift_phase2_{RUN_PROFILE}_{timestamp}"
    archive = Path(shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=RESULTS_ROOT.parent,
        base_dir=RESULTS_ROOT.name,
    ))
    checksum_path = archive.with_suffix(archive.suffix + ".sha256")
    checksum = sha256_file(archive)
    checksum_path.write_text(f"{checksum}  {archive.name}\n", encoding="utf-8")
    print({
        "archive": str(archive),
        "sha256": checksum,
        "checksum_file": str(checksum_path),
    })

## Faz 2 kapanış kriteri

`paper` profili tamamlandıktan sonra aşağıdakiler kontrol edilmelidir:

- Uniform sensitivity matrisi Cora + CiteSeer, GCN + GraphSAGE içeriyor.
- Aynı pruning artifact regression train/test tarafında paylaşılmıyor.
- `controls_only` ve `orbits_plus_controls` karşılaştırması raporlandı.
- Cross-dataset ve cross-model weight transfer sonuçları üretildi.
- Uniform–weighted farkları accuracy, Macro-F1 ve structural metriklerle raporlandı.
- Leave-one-orbit ve leave-group-out sonuçları üretildi.
- Guard ablation dört varyantla tamamlandı.
- `phase2_final_audit.json` hatasız üretildi.
- Sonuç zip'i ve SHA-256 dosyası Drive'a aktarıldı.

Negatif sonuçlar silinmemeli. Learned weighting uniform yöntemi geliştirmiyorsa bu da Faz 2'nin bilimsel sonucudur.